In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path(r"C:\local\supplyshield")
PROCESSED = ROOT / "data" / "processed"

data = pd.read_csv(PROCESSED / "legs_enriched.csv")

created = pd.to_datetime(data["trip_creation_time"], errors="raise")
departure = pd.to_datetime(data["od_start_time"], errors="raise")

rain = pd.to_numeric(data["source_rain_mm"], errors="coerce")
ratio = pd.to_numeric(data["delay_ratio"], errors="coerce")

# Use training trips and departure dates within the training window.
# This excludes later activity from trips created during training.
eligible = (
    created.ge("2018-09-12")
    & created.lt("2018-09-26")
    & departure.ge("2018-09-12")
    & departure.lt("2018-09-26")
    & rain.notna()
    & rain.ge(0)
    & ratio.notna()
    & ratio.ge(0)
    & ratio.lt(float("inf"))
)

analysis = data.loc[eligible].copy()
analysis["rain_mm"] = rain.loc[eligible]
analysis["ratio"] = ratio.loc[eligible]

analysis["condition"] = "light_rain_excluded"
analysis.loc[analysis["rain_mm"].eq(0), "condition"] = "dry"
analysis.loc[analysis["rain_mm"].ge(1), "condition"] = "rainy"

print("Eligible training legs with source weather:", len(analysis))
print("\nWeather groups:")
print(analysis["condition"].value_counts())

comparison_data = analysis.loc[
    analysis["condition"].isin(["dry", "rainy"])
]

keys = ["source_center", "destination_center"]

grouped = (
    comparison_data.groupby(keys + ["condition"])
    .agg(
        legs=("ratio", "size"),
        median_ratio=("ratio", "median")
    )
    .reset_index()
)

dry = grouped.loc[grouped["condition"].eq("dry")].rename(
    columns={"legs": "dry_legs", "median_ratio": "dry_median"}
)

rainy = grouped.loc[grouped["condition"].eq("rainy")].rename(
    columns={"legs": "rainy_legs", "median_ratio": "rainy_median"}
)

paired = dry[keys + ["dry_legs", "dry_median"]].merge(
    rainy[keys + ["rainy_legs", "rainy_median"]],
    on=keys,
    validate="one_to_one"
)

paired = paired.loc[
    paired["dry_legs"].ge(3) & paired["rainy_legs"].ge(3)
].copy()

paired["rainy_minus_dry"] = (
    paired["rainy_median"] - paired["dry_median"]
)

print("\nLanes with at least 3 dry and 3 rainy legs:", len(paired))

if paired.empty:
    print("Not enough comparable observations under this rule.")
else:
    print("Legs represented:",
          int((paired["dry_legs"] + paired["rainy_legs"]).sum()))
    print("Median within-lane difference:",
          round(paired["rainy_minus_dry"].median(), 4))
    print("Lanes with higher rainy-day median (%):",
          round(100 * paired["rainy_minus_dry"].gt(0).mean(), 2))
    display(paired.head(10))

Eligible training legs with source weather: 5118

Weather groups:
condition
rainy                  2156
light_rain_excluded    1483
dry                    1479
Name: count, dtype: int64

Lanes with at least 3 dry and 3 rainy legs: 113
Legs represented: 2179
Median within-lane difference: -0.0058
Lanes with higher rainy-day median (%): 47.79


,source_center,destination_center,dry_legs,dry_median,rainy_legs,rainy_median,rainy_minus_dry
0,IND000000AAL,IND411033AAA,3,2.500000,8,2.381944,-0.118056
1,IND000000ACA,IND144001AAB,6,1.609644,4,1.683824,0.074180
2,IND000000ACA,IND160002AAC,26,2.079877,16,2.225806,0.145929
3,IND000000ACB,IND110030AAD,14,1.682745,3,1.510204,-0.172541
4,IND000000ACB,IND110037AAK,15,1.675676,6,1.798077,0.122401
5,IND000000ACB,IND110064AAA,23,1.741379,8,1.456897,-0.284483
7,IND000000ACB,IND110095AAB,17,1.129870,6,1.730478,0.600608
8,IND000000ACB,IND121002AAB,24,2.038462,6,1.724208,-0.314253
9,IND000000ACB,IND121004AAB,15,1.818182,3,1.718750,-0.099432
10,IND000000ACB,IND122001AAA,24,1.691971,9,1.647059,-0.044913


In [2]:
paired.to_csv(
    PROCESSED / "rain_lane_comparison.csv",
    index=False
)

summary = pd.DataFrame([{
    "rain_threshold_mm": 1.0,
    "dry_definition": "exactly_0_mm",
    "minimum_legs_per_condition": 3,
    "eligible_training_legs": len(analysis),
    "comparable_lanes": len(paired),
    "represented_legs": int(
        (paired["dry_legs"] + paired["rainy_legs"]).sum()
    ),
    "median_within_lane_difference":
        paired["rainy_minus_dry"].median(),
    "lanes_with_positive_difference_pct":
        100 * paired["rainy_minus_dry"].gt(0).mean(),
}])

summary.to_csv(
    PROCESSED / "rain_comparison_summary.csv",
    index=False
)

print("Saved lane comparisons and summary.")

Saved lane comparisons and summary.


In [3]:
from pathlib import Path
import pandas as pd

ROOT = Path(r"C:\local\supplyshield")
PROCESSED = ROOT / "data" / "processed"

data = pd.read_csv(PROCESSED / "legs_enriched.csv")

created = pd.to_datetime(data["trip_creation_time"], errors="raise")
departure = pd.to_datetime(data["od_start_time"], errors="raise")
rain = pd.to_numeric(data["source_rain_mm"], errors="coerce")
ratio = pd.to_numeric(data["delay_ratio"], errors="coerce")

# Filter for the training window
eligible_mask = (
    created.ge("2018-09-12")
    & created.lt("2018-09-26")
    & departure.ge("2018-09-12")
    & departure.lt("2018-09-26")
    & rain.notna()
    & rain.ge(0)
    & ratio.notna()
    & ratio.ge(0)
    & ratio.lt(float("inf"))
)

base_df = data.loc[eligible_mask].copy()
base_df["rain_mm"] = rain.loc[eligible_mask]
base_df["ratio"] = ratio.loc[eligible_mask]

# Test multiple thresholds
thresholds = [1.0, 2.0, 5.0]
sensitivity_results = []

keys = ["source_center", "destination_center"]

for th in thresholds:
    df = base_df.copy()
    # Define condition based on threshold
    df["condition"] = "excluded"
    df.loc[df["rain_mm"].eq(0), "condition"] = "dry"
    df.loc[df["rain_mm"].ge(th), "condition"] = "rainy"
    
    comp_data = df.loc[df["condition"].isin(["dry", "rainy"])]
    
    grouped = (
        comp_data.groupby(keys + ["condition"])
        .agg(legs=("ratio", "size"), median_ratio=("ratio", "median"))
        .reset_index()
    )
    
    dry = grouped.loc[grouped["condition"].eq("dry")].rename(
        columns={"legs": "dry_legs", "median_ratio": "dry_median"}
    )
    rainy = grouped.loc[grouped["condition"].eq("rainy")].rename(
        columns={"legs": "rainy_legs", "median_ratio": "rainy_median"}
    )
    
    paired = dry[keys + ["dry_legs", "dry_median"]].merge(
        rainy[keys + ["rainy_legs", "rainy_median"]],
        on=keys,
        validate="one_to_one"
    )
    
    paired = paired.loc[paired["dry_legs"].ge(3) & paired["rainy_legs"].ge(3)].copy()
    paired["rainy_minus_dry"] = paired["rainy_median"] - paired["dry_median"]
    
    if not paired.empty:
        sensitivity_results.append({
            "threshold_mm": th,
            "comparable_lanes": len(paired),
            "represented_legs": int((paired["dry_legs"] + paired["rainy_legs"]).sum()),
            "median_within_lane_diff": paired["rainy_minus_dry"].median(),
            "pct_lanes_positive": 100 * paired["rainy_minus_dry"].gt(0).mean()
        })

sensitivity_summary = pd.DataFrame(sensitivity_results)
print(sensitivity_summary)

# Save sensitivity check output
sensitivity_summary.to_csv(PROCESSED / "rain_threshold_sensitivity.csv", index=False)
print("\nSaved sensitivity check summary.")

   threshold_mm  comparable_lanes  represented_legs  median_within_lane_diff  \
0           1.0               113              2179                -0.005760   
1           2.0               103              1819                -0.004825   
2           5.0                81              1471                -0.004825   

   pct_lanes_positive  
0           47.787611  
1           49.514563  
2           49.382716  

Saved sensitivity check summary.
